## Notebook complémentaire : bien choisir les opérateurs pour approximer connecteurs et quantificateurs

Ce notebook complète le tutoriel sur les opérateurs (`2-grounding_connectives.ipynb`).

On a vu que les connecteurs logiques sont groundés en LTN via une sémantique floue. Mais si n'importe quel opérateur flou valide a un sens quand on se contente d'**interroger** une formule déjà construite, tous ne se valent pas quand il s'agit d'**apprendre**, c'est-à-dire d'entraîner un modèle par descente de gradient à partir de cette formule.

On va voir ici les problèmes classiques que posent certaines sémantiques floues, et quels opérateurs sont mieux adaptés à l'apprentissage.

In [1]:
import ltn
import torch

### Querying

Le module `ltn.fuzzy_ops` donne accès à l'implémentation des sémantiques floues les plus courantes, construites à partir de primitives PyTorch.

On compare ici quatre opérateurs :
* la t-norme produit : $u \land_{\text{prod}} v = uv$ ;
* la t-norme de Łukasiewicz : $u \land_{\text{luk}} v = \max(u+v-1,\,0)$ ;
* l'agrégateur minimum : $\min(u_1,\dots,u_n)$ ;
* l'agrégateur `pMeanError` (moyenne généralisée des écarts à la vérité), qu'on a déjà rencontré et démontré : $\mathrm{pME}(u_1,\dots,u_n) = 1-\left(\frac1n\sum_{i=1}^n(1-u_i)^p\right)^{1/p}$.

Chacun de ces opérateurs porte évidemment un sens différent, et chacun peut avoir sa légitimité selon l'intention de la requête. On va voir dans les exemples suivants que des sémantiques différentes pour la conjonction donnent des résultats numériquement très différents, un phénomène qu'on retrouve aussi en comparant différents agrégateurs sur les mêmes valeurs d'entrée.

Prenons $u=0.4$ et $v=0.7$. Avec le produit, $u\land_{\text{prod}}v = 0.4\times0.7 = 0.28$. Avec Łukasiewicz, $u\land_{\text{luk}}v = \max(0.4+0.7-1,\,0) = \max(0.1,\,0) = 0.1$. Le résultat passe donc du simple au presque triple selon l'opérateur choisi, alors que les entrées sont identiques : c'est déjà un premier signal que le choix de la sémantique n'est pas neutre.

(Le paramètre `stable` utilisé dans le code ci-dessous sera expliqué à la fin de ce notebook.)

In [2]:
x1 = torch.tensor(0.4)
x2 = torch.tensor(0.7)

# the stable keyword is explained at the end of the notebook
and_prod = ltn.fuzzy_ops.AndProd(stable=False)
and_luk = ltn.fuzzy_ops.AndLuk()

print(and_prod(x1, x2))
print(and_luk(x1, x2))

tensor(0.2800)
tensor(0.1000)


On observe maintenant le même type d'écart, mais entre deux agrégateurs plutôt qu'entre deux conjonctions.

Prenons la suite $u = [1,\,1,\,1,\,0.5,\,0.3,\,0.2,\,0.2,\,0.1]$. Le minimum strict ne regarde que la pire valeur de la suite, ici $0.1$, et ignore complètement les sept autres. `pMeanError` avec $p=4$, en revanche, tient compte de toute la suite : le calcul donne environ $0.31$, un résultat sensiblement plus élevé que le minimum strict, précisément parce que les bonnes valeurs (les trois $1$) viennent partiellement compenser les mauvaises.

Cet écart illustre concrètement ce qu'on avait établi lors de la démonstration de convergence de `pMeanError` vers le minimum : avec un $p$ fini, l'agrégateur reste une version lissée du minimum, pas le minimum lui-même. Cette différence est justement au cœur de ce que ce notebook va explorer : pourquoi ce lissage est-il souhaitable pour l'entraînement, alors qu'il s'écarte de la sémantique logique stricte ?

In [3]:
xs = torch.tensor([1., 1., 1., 0.5, 0.3, 0.2, 0.2, 0.1])

# the stable keyword is explained at the end of the notebook
forall_min = ltn.fuzzy_ops.AggregMin()
forall_pME = ltn.fuzzy_ops.AggregPMeanError(p=4, stable=False)

print(forall_min(xs, dim=0))
print(forall_pME(xs, dim=0))

tensor(0.1000)
tensor(0.3134)


### Apprentissage

Si tous les opérateurs flous ont un sens quand on se contente d'interroger une formule déjà construite, ce n'est plus le cas dès qu'on veut **entraîner** un modèle à partir de cette formule. De nombreux opérateurs de logique floue ont des dérivées mal adaptées aux algorithmes fondés sur le gradient. Pour une analyse détaillée de ce phénomène, voir [van Krieken et al., *Analyzing Differentiable Fuzzy Logic Operators*, 2020](https://arxiv.org/abs/2002.06100).

On illustre ici, sur des cas simples, trois problèmes caractéristiques : le gradient qui s'annule, le gradient qui ne circule que sur une seule entrée à la fois, et le gradient qui explose.

#### 1. Le gradient qui s'annule (**Vanishing Gradient**)

Certains opérateurs ont un gradient nul sur une partie entière de leur domaine, ce qui bloque complètement l'apprentissage dans cette zone.

C'est le cas de la conjonction de Łukasiewicz, $u\land_{\text{luk}}v = \max(u+v-1,\,0)$. Dès que $u+v-1<0$, l'opérateur $\max$ renvoie $0$, et la dérivée de $\max(z,0)$ par rapport à $z$ vaut elle-même $0$ pour tout $z$ strictement négatif, pas seulement au point exact où la formule vaut $0$.

Avec $u=0.3$ et $v=0.5$, on a $u+v-1=-0.2<0$, donc $u\land_{\text{luk}}v=0$, et le gradient par rapport à $u$ comme à $v$ est exactement nul. Concrètement, cela signifie que l'algorithme d'optimisation ne reçoit **aucun signal** lui indiquant qu'il faudrait augmenter $u$ ou $v$ pour améliorer la satisfaction de la conjonction, alors que c'est pourtant intuitivement ce qu'il faudrait faire. Le gradient ne s'annule pas seulement en un point isolé, mais sur toute une région du domaine, ce qui peut bloquer l'apprentissage durablement dès que les valeurs y entrent.

In [4]:
x1 = torch.tensor(0.3, requires_grad=True)
x2 = torch.tensor(0.5, requires_grad=True)

y = and_luk(x1, x2)
y.backward()  # this is necessary to compute the gradients
res = y.item()
gradients = [v.grad for v in [x1, x2]]
# print the result of the aggregation
print(res)
# print gradients of x1 and x2
print(gradients)

0.0
[tensor(0.), tensor(0.)]


#### 2. Le gradient à passage unique (**Single-Passing Gradients**)

Certains opérateurs ne laissent circuler le gradient que vers une seule entrée à la fois, ce qui prive toutes les autres de tout signal d'apprentissage à cette étape.

C'est exactement le comportement du minimum, $\min(u_1,\dots,u_n)$, qu'on avait déjà anticipé lors de la démonstration sur `pMeanError` : la dérivée du minimum par rapport à $u_i$ vaut $1$ pour l'indice qui réalise effectivement le minimum, et $0$ pour tous les autres, quelle que soit leur valeur.

Sur la suite $u=[1,\,1,\,1,\,0.5,\,0.3,\,0.2,\,0.2,\,0.1]$, seul le dernier terme ($0.1$) réalise le minimum. Seul cet indice reçoit un gradient non nul, les sept autres valeurs, y compris celles qui sont loin d'être parfaites (comme $0.3$ ou $0.5$), ne reçoivent aucune mise à jour à cette étape. Dans un batch de grande taille, ce comportement est particulièrement inefficace : à chaque pas d'entraînement, un seul individu du batch progresse, tous les autres restent inchangés.

In [5]:
xs = torch.tensor([1., 1., 1., 0.5, 0.3, 0.2, 0.2, 0.1], requires_grad=True)

y = forall_min(xs, dim=0)
res = y.item()
y.backward()
gradients = xs.grad
# print the result of the aggregation
print(res)
# print gradients of xs
print(gradients)

0.10000000149011612
tensor([0., 0., 0., 0., 0., 0., 0., 1.])


#### 3. Le gradient qui explose (**Exploding Gradients**)

À l'inverse des deux cas précédents, certains opérateurs voient au contraire leur gradient devenir excessivement grand, voire diverger, sur certaines portions de leur domaine.

C'est le cas de l'agrégateur `pMeanError`, $\mathrm{pME}(u_1,\dots,u_n) = 1-\left(\frac1n\sum_{i=1}^n(1-u_i)^p\right)^{1/p}$, dans le cas limite où toutes les entrées valent exactement $1$. La somme à l'intérieur de la parenthèse devient alors nulle, puisque chaque terme $(1-u_i)$ s'annule, et l'expression prend la forme $0^{1/p}$. Or la dérivée d'une puissance fractionnaire $z^{1/p}$ (avec $p>1$) au voisinage de $z=0$ croît sans limite, précisément parce que l'exposant $1/p-1$ est négatif. Numériquement, cela se traduit par un gradient qui explose, voire une valeur non définie (`nan`), exactement au point où la formule est pourtant parfaitement satisfaite.

C'est une situation paradoxale : le cas le "meilleur" possible (tous les individus au maximum de vérité) devient numériquement instable pour l'entraînement, alors qu'on s'attendrait à ce que ce soit précisément le cas le plus simple à gérer.

In [6]:
xs = torch.tensor([1., 1., 1.], requires_grad=True)

y = forall_pME(xs, dim=0, p=4)
res = y.item()
y.backward()
gradients = xs.grad
# print the result of the aggregation
print(res)
# print the gradients of xs
print(gradients)

1.0
tensor([nan, nan, nan])


### Configuration produit stable

#### La configuration produit

On avait déjà recommandé, dans le notebook précédent, la configuration suivante, qu'on appelle la "configuration produit" :
* négation : la négation standard $\lnot u = 1-u$ ;
* conjonction : la t-norme produit $u\land v = uv$ ;
* disjonction : la t-conorme produit (somme probabiliste) $u\lor v = u+v-uv$ ;
* implication : l'implication de Reichenbach $u\implies v = 1-u+uv$ ;
* quantification existentielle : la moyenne généralisée `pMean`, $\mathrm{pM}(u_1,\dots,u_n)=\left(\frac1n\sum_{i=1}^n u_i^p\right)^{1/p}$ ;
* quantification universelle : la moyenne généralisée des écarts, `pMeanError`, $\mathrm{pME}(u_1,\dots,u_n)=1-\left(\frac1n\sum_{i=1}^n(1-u_i)^p\right)^{1/p}$.

Cette configuration, bien que recommandée, n'est pourtant pas totalement exempte des problèmes de gradient qu'on vient d'illustrer :
* la t-norme produit a un gradient qui s'annule dans le cas limite $u=v=0$ ;
* la t-conorme produit a un gradient qui s'annule dans le cas limite $u=v=1$ ;
* l'implication de Reichenbach a un gradient qui s'annule dans le cas limite $u=0,\,v=1$ ;
* `pMean` a un gradient qui explose dans le cas limite où tous les $u_i$ valent $0$ ;
* `pMeanError` a un gradient qui explose dans le cas limite où tous les $u_i$ valent $1$, exactement le problème qu'on vient d'observer numériquement.

#### La version "stable"

Ces problèmes ne surviennent que sur des cas limites bien identifiés, et se corrigent facilement grâce à l'astuce suivante :
* si le cas limite survient quand une entrée $u$ vaut $0$, on remplace chaque entrée par $u' = (1-\epsilon)u+\epsilon$ ;
* si le cas limite survient quand une entrée $u$ vaut $1$, on remplace chaque entrée par $u' = (1-\epsilon)u$ ;

où $\epsilon$ est une petite valeur positive (par exemple $10^{-5}$).

L'idée est simple : en décalant très légèrement les entrées pour qu'elles n'atteignent jamais exactement $0$ ou $1$, on évite mécaniquement le point précis où la dérivée s'annule ou explose, sans changer de façon perceptible le résultat de l'opérateur pour toutes les valeurs qui ne sont pas déjà sur ce cas limite. C'est cette version corrigée qu'on appelle "stable", au sens où elle ne présente plus ces problèmes de gradient.

On active la version stable d'un opérateur via le paramètre booléen `stable`, qu'on peut fixer par défaut à l'initialisation de l'opérateur, ou préciser différemment à chaque appel.

Reprenons l'exemple précédent, où toutes les entrées valaient $1$ et où le gradient de `pMeanError` explosait. En activant `stable=True`, on constate que les gradients ne sont plus `NaN` : la version stable de l'opérateur nous permet d'obtenir des gradients exploitables.

In [7]:
xs = torch.tensor([1., 1., 1.], requires_grad=True)

# the exploding gradient problem is solved
y = forall_pME(xs, dim=0, p=4, stable=True)
res = y.item()
y.backward()
gradients = xs.grad
# print the result of the aggregation
print(res)
# print the gradients of xs
print(gradients)

0.9998999834060669
tensor([0.3333, 0.3333, 0.3333])


#### L'hyperparamètre $p$ dans les moyennes généralisées

On a déjà vu que le paramètre $p$ de `pMean` et `pMeanError` permet d'écrire des formules plus ou moins strictes, pour tenir compte des valeurs aberrantes selon l'application. Ce paramètre doit cependant être choisi avec précaution, car il peut avoir des conséquences importantes sur l'entraînement.

On observe ci-dessous qu'une augmentation importante de $p$ conduit `pMeanError` à retomber dans le problème du gradient à passage unique. C'est cohérent avec ce qu'on avait démontré : `pMeanError` tend vers le minimum quand $p\to\infty$, et le minimum souffre exactement de ce défaut, vu plus haut dans ce notebook, à savoir que seul l'indice réalisant l'extremum reçoit un gradient non nul.

In [8]:
xs = torch.tensor([1., 1., 1., 0.5, 0.3, 0.2, 0.2, 0.1], requires_grad=True)

y = forall_pME(xs, dim=0, p=4)
res = y.item()
y.backward()
gradients = xs.grad
# print result of aggregation
print(res)
# print gradients of xs
print(gradients)

0.31339913606643677
tensor([0.0000, 0.0000, 0.0000, 0.0483, 0.1325, 0.1977, 0.1977, 0.2815])


In [9]:
xs = torch.tensor([1., 1., 1., 0.5, 0.3, 0.2, 0.2, 0.1], requires_grad=True)

y = forall_pME(xs, dim=0, p=20)
res = y.item()
y.backward()
gradients = xs.grad
# print result of aggregation
print(res)
# print gradients of xs
print(gradients)

0.18157517910003662
tensor([0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0734e-05, 6.4147e-03, 8.1100e-02,
        8.1100e-02, 7.6019e-01])


S'il peut être tentant de fixer un $p$ élevé pour obtenir des résultats logiquement stricts lors d'une simple interrogation, cela devient problématique en contexte d'apprentissage : un $p$ trop élevé transforme rapidement l'opérateur en un opérateur à passage unique, qui se concentre excessivement sur les valeurs extrêmes à chaque étape. Le gradient finit par surajuster une seule entrée du batch, au détriment de toutes les autres, ce qui peut nuire à l'entraînement global. Il est donc recommandé de ne pas fixer un $p$ trop élevé lorsqu'on entraîne un modèle, contrairement au cas où l'on se contente d'interroger une formule déjà construite.